# 01. Librerie

In [1]:
import pandas as pd
import numpy as np
import re

# 02. Input

In [7]:
input = r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\00_input"

df_occ = pd.read_excel(input + r"\Occupati_ISTAT.xlsx", sheet_name="Occupati", header=0)
df_inf = pd.read_excel(
    r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\02_output\INAIL_infortuni_NACE3.xlsx",
    sheet_name="infortuni")

# 03. Definizione Score

### Aggregazione infortuni in codici NACE 

In [8]:
# Funzioni utili
def melt_first_table(df_wide, year_col="AnnoAccadimento"):
    df_long = df_wide.melt(id_vars=[year_col], var_name="Codice_orig", value_name="Valore")
    df_long = df_long.rename(columns={year_col: "Anno"})
    df_long["Valore"] = pd.to_numeric(df_long["Valore"], errors="coerce").fillna(0)
    return df_long

_code_re = re.compile(r"^([A-Z])(?:[_\s-]?(\d+))?.*$")

def parse_code(code: str):

    m = _code_re.match(str(code).strip())
    if not m:
        return None, None
    letter = m.group(1)
    num = m.group(2)
    return letter, (int(num) if num is not None else None)

_range_re = re.compile(r"^([A-Z])[_\s-]?(\d+)\s*-\s*[A-Z]?[_\s-]?(\d+)$")

def build_rules(df_rules_wide, code_col="Codice_target"):

    rules = df_rules_wide[[code_col]].drop_duplicates().copy()
    rules[code_col] = rules[code_col].astype(str).str.strip()

    letters, da_list, a_list = [], [], []
    for c in rules[code_col]:

        rm = _range_re.match(c.replace("–", "-")) 
        if rm:
            letter = rm.group(1)
            da = int(rm.group(2))
            a = int(rm.group(3))
        else:
            letter, num = parse_code(c)
            if num is None:
                da = None
                a = None
            else:
                da = num
                a = num
        letters.append(letter)
        da_list.append(da)
        a_list.append(a)

    rules["Lettera"] = letters
    rules["Da"] = da_list
    rules["A"] = a_list
    return rules

def aggregate_to_rules(df1_long, rules):

    tmp = df1_long.merge(rules, on="Lettera", how="inner")

    is_letter_only = tmp["Da"].isna()
    in_range = tmp["Numero"].notna() & tmp["Da"].notna() & (tmp["Numero"] >= tmp["Da"]) & (tmp["Numero"] <= tmp["A"])
    tmp = tmp[is_letter_only | in_range]

    out = (tmp.groupby(["Anno", "Codice_target"], as_index=False)["Valore"]
             .sum()
             .rename(columns={"Valore": "Totale"}))
    return out

df1_wide = df_inf.copy()
df2 = df_occ.copy()

df2_codes = df2.rename(columns={"Nace": "Codice_target"})[["Codice_target"]]


df1_long = melt_first_table(df1_wide, year_col="AnnoAccadimento")
df1_long["Lettera"], df1_long["Numero"] = zip(*df1_long["Codice_orig"].map(parse_code))
rules = build_rules(df2_codes, code_col="Codice_target")
result = aggregate_to_rules(df1_long, rules)

result

      Anno Codice_target   Totale
0     2009          A_01   4334.0
1     2009          A_02    642.0
2     2009          A_03    291.0
3     2009             B   1228.0
4     2009     C_10-C_12  15161.0
...    ...           ...      ...
1003  2024          R_93   3615.0
1004  2024          S_94    968.0
1005  2024          S_95    502.0
1006  2024          S_96   3791.0
1007  2024             T     26.0

[1008 rows x 3 columns]


### Tabella pivot di infortuni

In [9]:
df_pivot = (
    result
    .pivot(index="Codice_target", columns="Anno", values="Totale")
    .sort_index()
    .sort_index(axis=1))

### Tabella tasso di infortuni su occupati

In [49]:
# pulizia 
df_3 = df2.drop(columns=['Tempo  ', '2008  '])
df_4 = df_3.rename(columns={"Nace": "Codice_target"})
df_5 = df_4.set_index("Codice_target")

df_5.columns = df_5.columns.astype(str).str.strip()
df_pivot.columns = df_pivot.columns.astype(str).str.strip()
df_5 = df_5.apply(pd.to_numeric, errors="coerce")

# rapporto
df_ratio = df_pivot / (df_5*1000) # Occupati in migliaia

df_ratio

Anno,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
Codice_target,,,,,,,,,,,,,,,,
A_01,0.004971,0.004994,0.004553,0.004280,0.003903,0.003746,0.003322,0.002953,0.003037,0.002873,0.003040,0.001296,0.001381,0.001403,0.001597,NaN
A_02,0.025375,0.026039,0.019021,0.015220,0.014712,0.011044,0.011053,0.010769,0.008549,0.008304,0.007681,0.007546,0.008868,0.009497,0.007500,NaN
A_03,0.010543,0.009621,0.009084,0.008045,0.007403,0.006846,0.004906,0.004796,0.003815,0.003602,0.003004,0.003896,0.003213,0.002941,0.002879,NaN
B,0.048538,0.048484,0.043554,0.034025,0.032435,0.028722,0.027032,0.030047,0.026714,0.028462,0.026845,0.016734,0.020208,0.020370,0.020000,0.019405
C_10-C_12,0.034116,0.033610,0.031174,0.028510,0.025987,0.024027,0.024883,0.024604,0.024996,0.024932,0.023933,0.021365,0.023366,0.023385,0.024233,0.024382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R_93,0.022639,0.021808,0.020746,0.019703,0.018088,0.016087,0.017182,0.017786,0.016949,0.015997,0.015509,0.013851,0.017079,0.017611,0.020000,NaN
S_94,0.011839,0.012315,0.012098,0.011746,0.011258,0.010472,0.009589,0.009462,0.010040,0.010335,0.010420,0.005536,0.007112,0.009694,0.006936,NaN
S_95,0.031540,0.028891,0.027665,0.025774,0.022524,0.020095,0.020469,0.018311,0.018973,0.017635,0.016548,0.009558,0.010802,0.011077,0.011151,NaN


### Calcolo media di settore NACE e normalizzazione massimo-minimo

In [62]:
# escludo 2024 per missing e lag
df_no2024 = df_ratio.drop(columns=["2024"], errors="ignore")

# media per codice NACE
mean_by_code = df_no2024.mean(axis=1, skipna=True)  

# output intermedio media
df_mean = mean_by_code.to_frame(name="media").T
df_mean.columns.name = "Codice_target"
df_mean.index = ["media"]

# normalizzazione (Min-max) ESCLUDENDO H_53 da min/max e forzando H_53=1
def row_minmax_exclude_outlier(s, outlier_code="H_53"):
    s_num = pd.to_numeric(s, errors="coerce")

    # min/max calcolati escludendo l'outlier
    s_wo = s_num.drop(labels=[outlier_code], errors="ignore")
    mn, mx = s_wo.min(), s_wo.max()

    # se non normalizzabile, ritorna NA (ma H_53 comunque 1 se presente)
    out = pd.Series([pd.NA] * len(s_num), index=s_num.index)
    if pd.notna(mn) and pd.notna(mx) and mx != mn:
        out = (s_num - mn) / (mx - mn)

    # impongo H_53 = 1
    if outlier_code in out.index:
        out.loc[outlier_code] = 1

    return out

df_norm = df_mean.apply(row_minmax_exclude_outlier, axis=1)
df_norm.index = ["Score"]

df_norm


Codice_target,A_01,A_02,A_03,B,C_10-C_12,C_13-C_15,C_16,C_17,C_18,C_19,...,O,P,Q_86,Q_87-Q_88,R_90-R_92,R_93,S_94,S_95,S_96,T
Score,0.051925,0.211508,0.093248,0.501085,0.435569,0.194432,0.571341,0.484185,0.319711,0.180694,...,0.223159,0.054057,0.512483,0.648967,0.1292,0.300111,0.164546,0.322277,0.150256,0.0


### Nace 3 digit valore di score

In [63]:
# lista codici NACE 3 digit
target_codes = [
"A_01","A_02","A_03","B_05","B_06","B_07","B_08","B_09","C_10","C_11","C_12","C_13","C_14","C_15",
"C_16","C_17","C_18","C_19","C_20","C_21","C_22","C_23","C_24","C_25","C_26","C_27","C_28","C_29",
"C_30","C_31","C_32","C_33","D_35","E_36","E_37","E_38","E_39","F_41","F_42","F_43","G_45","G_46",
"G_47","H_49","H_50","H_51","H_52","H_53","I_55","I_56","J_58","J_59","J_60","J_61","J_62","J_63",
"K_64","K_65","K_66","L_68","M_69","M_70","M_71","M_72","M_73","M_74","M_75","N_77","N_78","N_79",
"N_80","N_81","N_82","ND","O_84","P_85","Q_86","Q_87","Q_88","R_90","R_91","R_92","R_93","S_94",
"S_95","S_96","T_97","T_98","U_99"]

score_row_name = "Score"
scores_map = df_norm.loc[score_row_name].to_dict()

single_re = re.compile(r"^([A-Z])[_-](\d+)$")                # es: C_10
letter_re = re.compile(r"^([A-Z])$")                        # es: B
range_re  = re.compile(r"^([A-Z])[_-](\d+)\s*-\s*[A-Z]?[_-]?(\d+)$")  # es: C_10-C_12

def parse_single(code):
    m = single_re.match(str(code).strip())
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

ranges = [] 
for k, v in scores_map.items():
    kk = str(k).strip().replace("–", "-")
    m = range_re.match(kk)
    if m:
        letter = m.group(1)
        start = int(m.group(2))
        end   = int(m.group(3))
        if start > end:
            start, end = end, start
        ranges.append((k, letter, start, end, v))

def score_for_code(code):
    c = str(code).strip()

    # match unico
    if c in scores_map:
        return scores_map[c]

    # match "generico" di lettera (B -> B_05, B_06, ...)
    letter = c.split("_", 1)[0] if "_" in c else c
    if letter in scores_map:
        return scores_map[letter]

    # match intervallo (C_10-C_12 -> C_10, C_11, C_12)
    L, n = parse_single(c)
    if L is not None and n is not None:
        for _, rL, start, end, v in ranges:
            if rL == L and start <= n <= end:
                return v

    return pd.NA

# output
score_nace= pd.DataFrame(
    [[score_for_code(c) for c in target_codes]],
    index=[score_row_name],
    columns=target_codes
)

score_nace


,A_01,A_02,A_03,B_05,B_06,B_07,B_08,B_09,C_10,C_11,...,R_90,R_91,R_92,R_93,S_94,S_95,S_96,T_97,T_98,U_99
Score,0.051925,0.211508,0.093248,0.501085,0.501085,0.501085,0.501085,0.501085,0.435569,0.435569,...,0.1292,0.1292,0.1292,0.300111,0.164546,0.322277,0.150256,0.0,0.0,<NA>


In [64]:
# trasposizione
df_score = score_nace.T
df_score.columns = ["score"]

# aggiungi colonna Settore 
df_score["Settore"] = df_score.index.astype(str).str[0]

df_score = df_score.reset_index().rename(columns={"index": "Codice"})
df_score = df_score[df_score["Codice"] != "ND"]

df_score

,Codice,score,Settore
0,A_01,0.051925,A
1,A_02,0.211508,A
2,A_03,0.093248,A
3,B_05,0.501085,B
4,B_06,0.501085,B
...,...,...,...
84,S_95,0.322277,S
85,S_96,0.150256,S
86,T_97,0.0,T
87,T_98,0.0,T


## Export

In [65]:
# ---------- EXPORT ----------
out_file = r"C:\Users\fmeneghelli\OneDrive - KPMG\Tabacco, Filippo's files - 01_Indicatori_S_G\09_Infortuni_INAIL\99_Modelli\03.output_test\04_Definizione_score_var.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:

    # Dataframe completi
    df_score.to_excel(writer, sheet_name="Score_variabile", index=False)
    df_norm.to_excel(writer, sheet_name="Score_norm", index=False)
    df_ratio.to_excel(writer, sheet_name="DB_inf_ratio", index=True)